#### Extending the P0 Library

The task is to extend the P0 standard library with a function, `seed`, that initializes the random number generator below. The random number generator is used for the [Fermat primality test](https://en.wikipedia.org/wiki/Fermat_primality_test). According to Fermat, if `p` is prime and `a` is not divisible by `p`, then:

    a^(p − 1) mod p = 1

For a non-prime number `p`, this equality does not hold with high probability. This theorem is used for testing the primality of `p` by choosing `a` randomly. A chance that a composite number is reported as prime remains. That likelihood can be reduced by increasing the number of repetitions of the test, as specified by the parameter `k` of `likelyPrime(n, k)` below.  A *probabilistic* primality test, like the Fermat primality test, can be used to generate likely prime numbers for encryption. 

```Pascal
var r: integer

procedure randint(lower, upper: integer) → (rand: integer)
  // precondition:  lower < upper
  // postcondition: lower ≤ rand < upper
  const a = 16807
  const c = 11
  const m = 65535
    r := (a × r + c) mod m
    rand := r mod (upper - lower) + lower

procedure power(a: integer, n: integer, p: integer) → (res: integer)
  // precondition:  n ≥ 0 ∧ p > 1
  // postcondition: res = a₀^n₀ mod p, where a₀, n₀ are the initial values
  res, a := 1, a mod p
  // invariant: (res × (a^n mod p) = a₀^n₀ mod p) ∧ 0 ≤ res < p
  while n > 0 do
    if n mod 2 = 1 then // n odd
      res, n := (res × a) mod p, n - 1
    a, n := (a × a) mod p, n div 2
    
procedure likelyPrime(n: integer, k: integer)
  // precondition: n > 2 ∧ k > 0
  var i, p, a: integer
    i, p := k, 1
    while (i > 0) and (p = 1) do
      a ← randint(1, n - 1)
      p ← power(a, n - 1, n)
      i := i - 1
    if p = 1 then write(1) // prime
    else write(0) // composite

program primalityTest
  var n, k: integer
    r ← seed()
    n ← read(); k ← read()
    likelyPrime(n, k)
```

Library function `seed` should return the current time. Note that its result cannot be too large; otherwise, the multiplication with `a` may overflow. For example, the millisecond portion of the [current time](https://docs.python.org/3/library/datetime.html) works.

In [ ]:
# P0lib implementation in Python
def write(_: Machine, args: list[int]) -> list[int]:
    print(args[0]); return []
def writeln(_: Machine, args: list[int]) -> list[int]:
    print(); return []
def read(_: Machine, args: list[int]) -> list[int]:
    return [int(input())]
def seed(_: Machine, args: list[int]) -> list[int]:
    from datetime import datetime
    return [datetime.now().microsecond // 1000]

# Create runtime
from pywasm.core import Machine, Runtime, FuncType, ValType
runtime = Runtime()
runtime.imports = {'P0lib':
    {'write': runtime.allocate_func_host(FuncType([ValType.i32()], []), write),
     'writeln': runtime.allocate_func_host(FuncType([], []), writeln),
     'read': runtime.allocate_func_host(FuncType([], [ValType.i32()]), read),
     'seed': runtime.allocate_func_host(FuncType([], [ValType.i32()]), seed)}}

def runpywasm(wasmfile):
    # Create and run instance
    instance = runtime.instance_from_file(wasmfile)

<div style="float:right;background-color:lightgrey;border-left:20px solid white">

<pre>
;;  program setx
(func $program
  ;;  var x: integer
  (local $x i32)
  ;;    x := 0
  i32.const 0
  local.set $x
)
</pre>

</div>

<div style="float:right;background-color:lightgrey;border-left:20px solid white">

```Pascal
program setx
  var x: integer
    x := 0
```

</div>

Your WebAssembly code must follow the translation scheme from the course notes. In particular, it must call `read`, `write`, and `seed` as specified above. You may use the P0 compiler to compile fragments of the program. Every section of your WebAssembly code must be preceded by the corresponding P0 fragment, as illustrated to the right.

In [ ]:
%%writefile primality.wat
;;  var r: integer
(module
(import "P0lib" "write" (func $write (param i32)))
(import "P0lib" "writeln" (func $writeln))
(import "P0lib" "read" (func $read (result i32)))
(import "P0lib" "seed" (func $seed (result i32)))
(global $r (mut i32) i32.const 0)

;;  procedure randint(lower, upper: integer) → (rand: integer)
;;    const a = 16807
;;    const c = 11
;;    const m = 65535
;;      r := (a × r + c) mod m
;;      rand := r mod (upper - lower) + lower
(func $randint (param $lower i32) (param $upper i32) (result i32)
(local $rand i32)
(local $0 i32)
;;  r := (16807 × r + 11) mod 65535
i32.const 16807
global.get $r
i32.mul
i32.const 11
i32.add
i32.const 65535
i32.rem_s
global.set $r
;;  rand := r mod (upper - lower) + lower
global.get $r
local.get $upper
local.get $lower
i32.sub
i32.rem_s
local.get $lower
i32.add
local.set $rand
local.get $rand
)

;;  procedure power(a: integer, n: integer, p: integer) → (res: integer)
;;    res, a := 1, a mod p
;;    while n > 0 do
;;      if n mod 2 = 1 then
;;        res, n := (res × a) mod p, n - 1
;;      a, n := (a × a) mod p, n div 2
(func $power (param $a i32) (param $n i32) (param $p i32) (result i32)
(local $res i32)
(local $0 i32)
;;  res, a := 1, a mod p
i32.const 1
local.get $a
local.get $p
i32.rem_s
local.set $a
local.set $res
;;  while n > 0 do
loop
local.get $n
i32.const 0
i32.gt_s
if
;;    if n mod 2 = 1 then
local.get $n
i32.const 2
i32.rem_s
i32.const 1
i32.eq
if
;;      res, n := (res × a) mod p, n - 1
local.get $res
local.get $a
i32.mul
local.get $p
i32.rem_s
local.get $n
i32.const 1
i32.sub
local.set $n
local.set $res
end
;;    a, n := (a × a) mod p, n div 2
local.get $a
local.get $a
i32.mul
local.get $p
i32.rem_s
local.get $n
i32.const 2
i32.div_s
local.set $n
local.set $a
br 1
end
end
local.get $res
)

;;  procedure likelyPrime(n: integer, k: integer)
;;    var i, p, a: integer
;;      i, p := k, 1
;;      while (i > 0) and (p = 1) do
;;        a ← randint(1, n - 1)
;;        p ← power(a, n - 1, n)
;;        i := i - 1
;;      if p = 1 then write(1) else write(0)
(func $likelyPrime (param $n i32) (param $k i32)
(local $i i32)
(local $p i32)
(local $a i32)
(local $0 i32)
;;  i, p := k, 1
local.get $k
i32.const 1
local.set $p
local.set $i
;;  while (i > 0) and (p = 1) do
loop
local.get $i
i32.const 0
i32.gt_s
if (result i32)
local.get $p
i32.const 1
i32.eq
else
i32.const 0
end
if
;;    a ← randint(1, n - 1)
i32.const 1
local.get $n
i32.const 1
i32.sub
call $randint
local.set $a
;;    p ← power(a, n - 1, n)
local.get $a
local.get $n
i32.const 1
i32.sub
local.get $n
call $power
local.set $p
;;    i := i - 1
local.get $i
i32.const 1
i32.sub
local.set $i
br 1
end
end
;;  if p = 1 then write(1) else write(0)
local.get $p
i32.const 1
i32.eq
if
i32.const 1
call $write
else
i32.const 0
call $write
end
)

;;  program primalityTest
;;    var n, k: integer
;;      r ← seed()
;;      n ← read(); k ← read()
;;      likelyPrime(n, k)
(global $_memsize (mut i32) i32.const 0)
(func $program
(local $n i32)
(local $k i32)
(local $0 i32)
;;  r ← seed()
call $seed
global.set $r
;;  n ← read()
call $read
local.set $n
;;  k ← read()
call $read
local.set $k
;;  likelyPrime(n, k)
local.get $n
local.get $k
call $likelyPrime
)
(memory 1)
(start $program)
)

In [ ]:
!wat2wasm primality.wat

You can test your solution interactively by uncommenting the following cell. However, you must comment out or delete any code that runs interactively before submitting, as otherwise grading will fail.

In [ ]:
# runpywasm("primality.wasm")

Your solution is being graded on how well it tests for primality.

The cells below are for grading purposes only; please ignore them.

In [ ]:
%%capture output

In [ ]:
%%capture output